# Experiment 4 (Advanced): Baseline Review

**Learning objective:** judge whether a test-time-compute paper's experimental baselines are fair, meaning whether the Self-Consistency-style aggregation is reproduced correctly, whether the budget-range coverage is complete, whether the verifier training setup is clearly described, and whether the difficulty bucketing is applied fairly.

**Your task:** critically review the experiment design of **Scaling LLM Test-Time Compute Optimally (ICLR 2025)** and produce a short report answering the four mandatory questions below. The paper PDF is included with this course at `references/02-Scaling_LLM_Test-Time_Compute_Optimally.pdf`; focus on the abstract, Section 3.2, and Appendices E/K/N/O.

## The Four Mandatory Questions

1. **Is the budget-range coverage complete** (from small to large budgets)?

2. **Are the majority-voting (Self-Consistency-style aggregation) and best-of-N weighted baselines faithful to their original definitions** (number of samples, score-based selection)? **Is the paper's difficulty bucketing applied fairly?**

   Note: the paper's majority voting is Self-Consistency-style aggregation, but it never uses that name nor cites the original SC paper.

3. **Is the verifier (PRM) training setup clearly described?**

4. **Point out at least one baseline flaw.**

## Suggested Report Structure

- **Abstract:** one sentence answering whether the paper's experimental baseline is fair.
- **Q1: Budget coverage:** budget range the paper actually covers vs. the range its conclusions claim
- **Q2: Baseline fidelity:** check sampling counts and score selection against the original definitions (Self-Consistency-style aggregation; best-of-N), and whether the difficulty bucketing is applied fairly
- **Q3: Verifier setup:** is the PRM training data, procedure, and sampling protocol reproducible from the paper and its appendix?
- **Q4: Baseline flaw:** at least one, with its impact on the conclusions


---

## 1. (Optional) Export the Course Data for Comparison

Export the headline table of this course so you can contrast our setup (weak verifier, B=50, GGUF Q8_0) with the paper's (strong PRM, FLOPs budget, unquantized weights).


In [ ]:
# Export the headline data table
import os, subprocess, sys, glob, json

# Locate the repository root and switch to it (see Notebook 01)
_REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_REPO_ROOT, "scripts", "run_experiment.py")):
    _parent = os.path.dirname(_REPO_ROOT)
    if _parent == _REPO_ROOT:
        break
    _REPO_ROOT = _parent
if os.path.abspath(os.getcwd()) != _REPO_ROOT:
    os.chdir(_REPO_ROOT)
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

# Export the headline table - your own data if you ran the experiments,
# otherwise the included cache layer (data/cache_subset/, 6 models on MATH-500 + 2 on GSM8K).
inputs = sorted(glob.glob(os.path.join("data", "results", "experiment1_*", "exp1.json")))
print("Available local experiment files:", inputs)
if inputs:
    out = os.path.join("data", "results")
    cmd = [sys.executable, "scripts/export_data.py",
           "--inputs"] + inputs + ["--out-dir", out]
    try:
        subprocess.run(cmd, check=True)
        print("Exported:", [os.path.basename(p) for p in glob.glob(os.path.join(out, "headline_*"))])
    except Exception as e:
        print("Export failed:", type(e).__name__, str(e)[:200])
else:
    # Cache layer: show the included headline data (six models on MATH-500 + two on GSM8K)
    hl_path = os.path.join("data", "cache_subset", "headline.json")
    if os.path.exists(hl_path):
        print("%-34s %-8s %s" % ("model", "dataset", "N=50/M=0 | N=25/M=1 | N=5/M=9"))
        for h in json.load(open(hl_path, encoding="utf-8"))["headline"]:
            vals = []
            for k in ("N=50,M=0", "N=25,M=1", "N=5,M=9"):
                rec = h["headline"][k]
                vals.append("%.3f" % rec["accuracy"] if rec else "-")
            print("%-34s %-8s %s" % (h["model"], h["dataset"], " | ".join(vals)))
        print("(Included cache: 100 problems, seed 0, GGUF Q8_0 - 3-seed headline numbers are in the notebook intro.)")
    else:
        print("Note: no local experiment files and no cache found - run Notebook 01 first.")

---

## 2. Common Misconceptions (check yourself)

These three claims first appeared in Notebooks 01/02 (and self-check question 2); the recap below adds the measured numbers.

1. **"More verification is always more accurate" is false.** With a weak verifier (alpha ≈ 0.62, beta ≈ 0.37), verifier quality sets the ceiling on verification, not the number of checks M. Both are measured from Qwen3-0.6B self-scoring runs (100-problem seed-0, scores on a 1-10 scale). Watch whether the verify-heavy split (N=5/M=9) beats the verify-light split (N=10/M=4) at the same 50-call budget: in our 100-problem data the two splits land within noise of each other and the winner flips by model (-5 to +6 pp).

2. **"Voting always helps" is false.** The mathematical guarantee (Chernoff) holds only when p>1/2. With p<=1/2, voting can hurt when errors are concentrated (self-check question 2: 21.6% < 30%) or still help when errors are dispersed (measured +7.3 pp at p ≈ 0.37).

3. **"One inference can be parallelized into many" is false.** N candidates are generated by N *independent* LLM calls; parallelism changes wall-clock time only, not the call budget N x (1+M) = B.

---

## 3. Write Your Critical Report

Use the structure above. This is the main deliverable: make it specific. Cite concrete numbers from the paper (abstract, Section 3.2, Appendix E on difficulty bucketing, and Appendices K/N/O) and from the data export in Section 1.


In [ ]:
# Your report goes here (markdown) - or write it in the cell below as text.
# Suggested length: 300-500 words. Remember Q4: at least one concrete flaw.
REPORT_DRAFT = """
Abstract: ...

Q1 - Budget coverage: ...
Q2 - Baseline fidelity: ...
Q3 - Verifier setup: ...
Q4 - Baseline flaw: ...
"""
print("Report draft ready. (Fill in the text above before submission.)")

---

## (For Instructors) Reference Answers

The instructor reference answers are in the cell below, in plain text. They are an honesty mechanism: write your own report before peeking. The instructor can use them to check completeness; the key grading criterion is that each of the four questions is answered with concrete evidence, and Q4 names at least one real flaw.


In [ ]:
# INSTRUCTOR REFERENCE ANSWERS - source contains the answers, so write your own
# report first (self-discipline). Runs only if INSTRUCTOR=1 is set in the environment.
import os
if os.environ.get("INSTRUCTOR") != "1":
    print("Instructor-only cell. Students: please write your own report instead.")
else:
    print("""
Instructor reference answers (baseline review):

Q1 - Budget coverage: the paper scans small to large budgets (full spectrum in
the appendix), but the main figures focus on the moderate range; the '4x
efficiency' headline comes from compute-optimal *adaptive* allocation (abstract
wording) - check whether the comparison to fixed ratios holds within the same
budget range. If the main text covers only part of the range, extrapolation
needs caution.

Q2 - Baseline fidelity: best-of-N weighted uses PRM-score-weighted selection
(depends on verifier quality); Majority voting is a parallel-sampling
aggregation baseline (the paper does not name it Self-Consistency nor cite the
original paper, but the mechanism is equivalent) - check that the reproduction
uses the same sampling count N, the same PRM for score selection, and a
faithful difficulty bucketing (pass@1 quantiles, paper Section 3.2).

Q3 - Verifier setup: the paper's PRM is a specially trained strong verifier
(reward model / process supervision); its training data, method, and sampling
protocol must be fully reproducible from the paper and appendix - if settings
are missing or inconsistent with the main experiments, comparability of the
verification-based conclusions is questionable.

Q4 - Baseline flaw (example): best-of-N depends on a strong verifier while
Majority does not, so the comparison does not isolate the verifier-quality
factor (the equal-candidate-count control in our Experiment 1 is designed
exactly for this).
    """)

---

## Summary

- You reviewed the ICLR 2025 baseline design against four mandatory questions and produced a critical report.
- The course data export lets you contrast setups (weak verifier, B=50, Q8_0 vs. strong PRM, FLOPs budget, unquantized weights), which is itself material for the report.

This completes the notebook series. Well done!
